# Демонстрационный ноутбук для `scripts/main.py`

Задача эксперимента: проверить, может ли мета-модель, обученная на одних текстовых датасетах, рекомендовать размерность embedding для новых unseen-датасетов без существенной потери качества.
Гипотеза: предсказанная размерность сохраняет не менее 95% базового качества классификации при заметном снижении размерности.
Критерий успеха: сжатая модель должна показывать качество >= 95% baseline на новых датасетах.


In [ ]:
# Импорт необходимых библиотек
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from dataset_complexity_profiler import DatasetProfiler


## Вспомогательные функции

Здесь определены функции для загрузки текстовых датасетов, оценки качества классификации на полном embedding, а также для подготовки данных и предсказания размерности.
Это помогает сделать эксперимент понятным и воспроизводимым.


In [ ]:
def load_dotenv(dotenv_path: str = ".env") -> None:
    if os.environ.get("HF_TOKEN"):
        return
    path = Path(dotenv_path)
    if not path.is_file():
        return

    with path.open("r", encoding="utf-8") as env_file:
        for line in env_file:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip()
            if (value.startswith('"') and value.endswith('"')) or (value.startswith("'") and value.endswith("'")):
                value = value[1:-1]
            if key and value and key not in os.environ:
                os.environ[key] = value

def load_text_dataset(dataset_id: str, split: str = "train", sample_limit: int = 600):
    token = os.environ.get("HF_TOKEN")
    kwargs = {"split": split}
    if token:
        kwargs["token"] = token
    dataset = load_dataset(dataset_id, **kwargs)

    if sample_limit and len(dataset) > sample_limit:
        dataset = dataset.shuffle(seed=42).select(range(sample_limit))

    text_column = None
    for candidate in ["text", "sentence", "content", "review"]:
        if candidate in dataset.column_names:
            text_column = candidate
            break
    if text_column is None:
        raise ValueError(f"No text column found in dataset {dataset_id}")

    if "label" in dataset.column_names:
        label_column = "label"
    elif "labels" in dataset.column_names:
        label_column = "labels"
    else:
        raise ValueError(f"No label column found in dataset {dataset_id}")

    return list(dataset[text_column]), list(dataset[label_column])

def evaluate_embedding_quality(X: np.ndarray, y) -> float:
    if X.shape[1] == 0:
        return 0.0
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )
    except ValueError:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=None
        )

    classifier = LogisticRegression(max_iter=2000, solver="lbfgs")
    classifier.fit(X_train, y_train)
    return float(classifier.score(X_test, y_test))

def predict_dataset(dataset_id: str, split: str = "train", sample_limit: int = 600):
    texts, labels = load_text_dataset(dataset_id, split, sample_limit)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    X = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)

    profiler = DatasetProfiler(auto_load_meta_model=False)
    profiler.load_default_meta_model()
    predicted_dim = profiler.predict_embedding_dim(X, labels)
    architecture = profiler.recommend_architecture(
        X.shape[1], predicted_dim, n_classes=len(set(labels))
    )

    return {
        "dataset_id": dataset_id,
        "sample_count": len(labels),
        "original_dim": int(X.shape[1]),
        "class_count": len(set(labels)),
        "predicted_embedding_dim": int(predicted_dim),
        "architecture_recommendation": architecture,
    }


## Обучение мета-модели и проверка на новых датасетах

Здесь мета-модель обучается на предварительно выбранных seen-датасетах, а затем проверяется на holdout-датасетах, которых не было в обучении.
Это демонстрирует способность метода обобщать на новые задачи.


In [ ]:
def evaluate_dataset_prediction(profiler, texts, labels, quality_threshold: float = 0.95):
    # Получаем исходные embeddings и baseline-качество
    X = profiler.embed_texts(texts, embedder_name="all-MiniLM-L6-v2", batch_size=64, show_progress=False)
    baseline_quality = evaluate_embedding_quality(X, labels)
    # Оцениваем оптимальную размерность, сохраняющую требуемое качество
    estimate_info = profiler.estimate_intrinsic_dim(X, labels, quality_threshold=quality_threshold)
    true_dim = int(estimate_info["recommended_dim"])
    predicted_dim = profiler.predict_embedding_dim(X, labels)

    # Сжимаем embeddings до предсказанной размерности
    X_pred = PCA(n_components=min(predicted_dim, X.shape[1], X.shape[0] - 1)).fit_transform(X)
    compressed_quality = evaluate_embedding_quality(X_pred, labels)

    # Сравниваем с истинной рекомендованной размерностью
    X_true = PCA(n_components=min(true_dim, X.shape[1], X.shape[0] - 1)).fit_transform(X)
    true_quality = evaluate_embedding_quality(X_true, labels)

    original_dim = X.shape[1]
    compression_ratio = float(round((1 - predicted_dim / original_dim) * 100, 2))
    quality_ratio = float(round(compressed_quality / baseline_quality, 4)) if baseline_quality > 0 else 0.0
    success = compressed_quality >= baseline_quality * quality_threshold

    return {
        "baseline_model_quality": float(round(baseline_quality, 4)),
        "true_optimal_dim": true_dim,
        "predicted_dim": int(predicted_dim),
        "predicted_model_quality": float(round(compressed_quality, 4)),
        "true_optimal_dim_quality": float(round(true_quality, 4)),
        "quality_ratio": quality_ratio,
        "compression_ratio": compression_ratio,
        "success": success,
        "baseline_dim": original_dim,
    }


In [ ]:
load_dotenv()

seen_datasets = [
    ("mteb/NewsClassification", "train"),
    ("mteb/SentimentAnalysisHindi", "train"),
]
holdout_datasets = [
    ("mteb/emotion", "train"),
    ("mteb/SentimentDKSF", "train"),
]

print("Загружаем seen-датасеты для обучения мета-модели...")
training_sets = []
for dataset_id, split in seen_datasets:
    texts, labels = load_text_dataset(dataset_id, split, sample_limit=300)
    training_sets.append({"texts": texts, "labels": labels})
    print(f"  Загружено {len(texts)} примеров из {dataset_id}")

profiler = DatasetProfiler(auto_load_meta_model=False)
profiler.train_custom_meta_model(
    training_sets,
    quality_threshold=0.95,
    embedder_name="all-MiniLM-L6-v2",
    batch_size=64,
    show_progress=False,
    cv=3,
)
print("Мета-модель обучена на seen-датасетах.")


## Проверка на holdout-датасетах

Теперь проверяем качество предсказаний на новых датасетах, которые не участвовали в обучении.
Для каждого датасета сравниваем baseline-качество, качество после сжатия и фактическое снижение размерности.


In [ ]:
print("Оцениваем holdout-датасеты...")
results = []
for dataset_id, split in holdout_datasets:
    texts, labels = load_text_dataset(dataset_id, split, sample_limit=300)
    print(f"  Оцениваем {dataset_id} ({len(texts)} примеров)...")
    eval_report = evaluate_dataset_prediction(profiler, texts, labels, quality_threshold=0.95)
    eval_report["dataset_id"] = dataset_id
    results.append(eval_report)
print("Проверка завершена.")


## Результаты проверки на holdout-датасетах

В таблице показаны метрики для каждого unseen-датасета.
- baseline-качество на полном embedding
- предсказанная размерность
- качество после сжатия
- отношение качества к baseline
- процент снижения размерности
- факт успешного предсказания (>= 95% baseline)


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df[[
    "dataset_id",
    "baseline_model_quality",
    "predicted_dim",
    "true_optimal_dim",
    "predicted_model_quality",
    "true_optimal_dim_quality",
    "quality_ratio",
    "compression_ratio",
    "success",
]]
display(results_df)

summary = {
    "avg_quality_ratio": float(round(results_df["quality_ratio"].mean(), 4)),
    "avg_compression": float(round(results_df["compression_ratio"].mean(), 2)),
    "success_rate": float(round(results_df["success"].mean() * 100, 1)),
}

print("=== Итоговые метрики ===")
print(f"Среднее сохраненное качество: {summary['avg_quality_ratio']:.4f}")
print(f"Среднее снижение размерности: {summary['avg_compression']:.2f}%")
print(f"Доля успешных предсказаний: {summary['success_rate']:.1f}%")


## Вывод

- Мета-модель, обученная на seen-датасетах, может предсказывать размерность embedding для новых unseen-датасетов и сохранять качество близкое к baseline.
- Важно, что проверка проводится на holdout-датасетах, которых не было в обучении, поэтому результаты демонстрируют обобщающую способность.
- Ограничение демо: используется ограниченное число текстовых задач и PCA для моделирования сжатия, но сам подход показывает практический путь к автоматическому выбору размерности.
